# Module 07: Retrieval-Augmented Generation (RAG) Systems

**Companion notebook for**: `07-rag-systems.html`

## Learning Objectives
- Understand the RAG pipeline: indexing phase (load, chunk, embed, store) and query phase (embed, retrieve, generate)
- Load documents from plain text and understand PDF loading concepts
- Compare text chunking strategies: fixed-size, recursive character, and semantic
- Generate embeddings with OpenAI and compute cosine similarity
- Set up an in-memory ChromaDB vector store and perform similarity search
- Build a complete end-to-end RAG pipeline with source citations
- Evaluate retrieval quality and manage context windows
- Implement a RAG chain using LangChain

## Prerequisites
- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q openai chromadb langchain langchain-openai langchain-community tiktoken numpy

In [ ]:
import os
import json
import textwrap
from pprint import pprint

import numpy as np
from openai import OpenAI

# Verify API key
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"
client = OpenAI()
print("All imports successful. OpenAI API key found.")

---
## Section 1: RAG Fundamentals

Retrieval-Augmented Generation (RAG) is the most impactful pattern in production GenAI. Instead of relying solely on what the model memorized during training, RAG **retrieves relevant documents** from your own data and injects them into the prompt, giving the model up-to-date, domain-specific knowledge.

RAG solves three core problems:
1. **Knowledge cutoff** -- the model cannot know about events after its training date
2. **Hallucination** -- without evidence, LLMs fabricate plausible-sounding answers
3. **Domain specificity** -- general-purpose LLMs lack knowledge of your internal data

### The Two Phases

| Phase | When | Steps |
|-------|------|-------|
| **Indexing** | Offline (once) | Load docs -> Chunk -> Embed -> Store in vector DB |
| **Query** | Real-time | Embed query -> Retrieve top-K chunks -> LLM generates answer |

In mathematical terms, a standard LLM computes **P(y|x)**. RAG modifies this to **P(y|x, z)** where *z* is the retrieved context -- the model's output is now conditioned on actual evidence.

---
## Section 2: Document Loading

The first step is extracting text from source documents. In production you would handle PDF, DOCX, HTML, and database sources. Here we demonstrate the core pattern with plain text, then show how LangChain loaders generalize to other formats.

In [ ]:
# --- Sample knowledge base (plain text documents) ---
# In production these would be loaded from files. We inline them here
# so the notebook is fully self-contained.

SAMPLE_DOCUMENTS = [
    {
        "text": (
            "Retrieval-Augmented Generation (RAG) Overview\n\n"
            "RAG is a technique that enhances LLM outputs by retrieving relevant "
            "documents from an external knowledge base and injecting them into the "
            "prompt. This grounds the model's response in actual source material "
            "rather than relying solely on memorized training data.\n\n"
            "The RAG pipeline has two main phases. The indexing phase happens offline: "
            "documents are loaded, split into chunks, converted into embedding vectors, "
            "and stored in a vector database. The query phase happens in real time: the "
            "user's question is embedded, similar chunks are retrieved via vector search, "
            "and those chunks are passed to the LLM as context for answer generation.\n\n"
            "RAG dramatically reduces hallucination because the model has actual source "
            "text to reference. It also keeps answers current -- updating the knowledge "
            "base is as simple as adding new documents, with no model retraining needed."
        ),
        "source": "rag_overview.txt",
        "page": 1,
    },
    {
        "text": (
            "Embedding Models for RAG\n\n"
            "An embedding model converts text into a dense numerical vector that captures "
            "semantic meaning. Similar texts produce vectors that are close together in "
            "the embedding space, enabling semantic search -- matching by meaning rather "
            "than exact keywords.\n\n"
            "Popular embedding models include OpenAI text-embedding-3-small (1536 dims, "
            "$0.02/1M tokens), text-embedding-3-large (3072 dims, $0.13/1M tokens), and "
            "open-source alternatives like BAAI/bge-large-en-v1.5 (1024 dims, free).\n\n"
            "A critical rule: you must use the same embedding model for indexing and "
            "querying. Each model maps text to its own coordinate system, so vectors "
            "from different models are not comparable."
        ),
        "source": "embeddings_guide.txt",
        "page": 1,
    },
    {
        "text": (
            "Vector Databases\n\n"
            "A vector database stores embedding vectors and supports fast approximate "
            "nearest neighbor (ANN) search. Unlike traditional databases optimized for "
            "exact matches, vector databases find the K most similar vectors out of "
            "millions using specialized indexing structures like HNSW.\n\n"
            "ChromaDB is the simplest option -- an in-process database requiring no "
            "separate server. Pinecone is a fully managed cloud service. pgvector adds "
            "vector search to PostgreSQL. Qdrant and Weaviate are purpose-built for "
            "advanced features like filtering and hybrid search.\n\n"
            "Start with ChromaDB for prototyping and migrate to a managed solution "
            "when you hit scale limits."
        ),
        "source": "vector_db_guide.txt",
        "page": 1,
    },
    {
        "text": (
            "Chunking Strategies\n\n"
            "Chunking splits large documents into smaller, semantically coherent pieces "
            "for embedding and retrieval. The goal is chunks that are self-contained -- "
            "each makes sense on its own.\n\n"
            "Fixed-size chunking splits text every N characters regardless of content. "
            "It is fast but often cuts sentences in half. Recursive character splitting "
            "tries natural boundaries first: paragraph breaks, then line breaks, then "
            "sentences, then words. Semantic chunking uses embedding similarity to detect "
            "topic changes and split at those boundaries.\n\n"
            "Overlap (10-20% of chunk size) ensures information spanning a boundary "
            "appears in at least one complete chunk. Start with 512 tokens and 20% overlap."
        ),
        "source": "chunking_guide.txt",
        "page": 1,
    },
    {
        "text": (
            "Fine-Tuning vs RAG\n\n"
            "Fine-tuning modifies the model's weights using domain-specific training "
            "data. It is expensive, requires GPU compute, and the knowledge is baked "
            "into the model permanently. Updating requires retraining.\n\n"
            "RAG keeps the model unchanged and injects knowledge at query time via "
            "retrieval. It is cheaper, easier to update, and provides citations. "
            "However, RAG is limited by context window size and retrieval quality.\n\n"
            "In practice, most production systems use RAG as the default and only "
            "fine-tune when RAG cannot achieve the required performance -- for example, "
            "when the task requires a specific output format or writing style."
        ),
        "source": "fine_tuning_vs_rag.txt",
        "page": 1,
    },
    {
        "text": (
            "Retrieval Strategies\n\n"
            "Pure vector search uses cosine similarity to find the K most similar "
            "chunks to the query embedding. It excels at semantic matching but can "
            "miss exact terms like product codes or error messages.\n\n"
            "Hybrid search combines vector similarity with BM25 keyword matching "
            "using Reciprocal Rank Fusion (RRF). This captures both semantic meaning "
            "and exact keyword matches.\n\n"
            "Reranking is a second stage that uses a cross-encoder model to rescore "
            "the initial candidates. Cross-encoders process query and document together "
            "for much better relevance judgments, but are too slow to run against the "
            "full collection. Adding a reranker typically improves precision by 15-30%."
        ),
        "source": "retrieval_guide.txt",
        "page": 1,
    },
    {
        "text": (
            "LLM Synthesis and Prompt Design\n\n"
            "The synthesis step combines the user question with retrieved context in a "
            "carefully structured prompt. The system prompt instructs the model to base "
            "its answer only on the provided context and to say 'I don't know' when the "
            "context is insufficient.\n\n"
            "Use temperature 0.0 to 0.2 for RAG synthesis. Higher temperatures encourage "
            "creative generation, which works against factual grounding. Low temperature "
            "makes the model stick closely to the provided evidence.\n\n"
            "Include source metadata in the context so the model can provide citations. "
            "Format context with clear delimiters -- numbered sections or XML tags -- so "
            "the model can distinguish between different source documents."
        ),
        "source": "synthesis_guide.txt",
        "page": 1,
    },
]

print(f"Loaded {len(SAMPLE_DOCUMENTS)} sample documents")
for doc in SAMPLE_DOCUMENTS:
    print(f"  - {doc['source']} ({len(doc['text'])} chars)")

In [ ]:
# --- Document loading patterns for real-world formats ---
# These demonstrate the LangChain loader API; they require actual files
# on disk, so we show the code but do not execute it.

LOADER_EXAMPLES = '''
from langchain_community.document_loaders import (
    PyPDFLoader,                      # PDF files
    UnstructuredWordDocumentLoader,    # DOCX files
    UnstructuredHTMLLoader,            # HTML files
    TextLoader,                        # Plain text / Markdown
)
from pathlib import Path

def load_documents(directory: str) -> list:
    """Load all documents from a directory, auto-detecting format."""
    loaders = {
        ".pdf":  PyPDFLoader,
        ".docx": UnstructuredWordDocumentLoader,
        ".html": UnstructuredHTMLLoader,
        ".txt":  TextLoader,
        ".md":   TextLoader,
    }
    docs = []
    for path in Path(directory).rglob("*"):
        loader_cls = loaders.get(path.suffix.lower())
        if loader_cls:
            loader = loader_cls(str(path))
            docs.extend(loader.load())
    return docs

docs = load_documents("./knowledge_base")
'''

print("Document Loader Reference (not executed -- requires files on disk):")
print(LOADER_EXAMPLES)

---
## Section 3: Text Chunking Strategies

Chunking splits documents into smaller pieces for embedding and retrieval. The right chunk size balances:
- **Too small**: loses context (a sentence in isolation may not make sense)
- **Too large**: wastes context window space with irrelevant text

We compare three strategies:
1. **Fixed-size** -- split every N characters
2. **Recursive character** -- split on natural boundaries (paragraphs > lines > sentences > words)
3. **Semantic** -- split where embedding similarity drops (topic change)

In [ ]:
# Use one of our sample documents for chunking demonstrations
sample_text = SAMPLE_DOCUMENTS[0]["text"]
print(f"Sample text length: {len(sample_text)} characters")
print(f"Preview: {sample_text[:200]}...")

In [ ]:
# --- Strategy 1: Fixed-Size Chunking ---

def fixed_size_chunk(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    """Split text into fixed-size chunks with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

fixed_chunks = fixed_size_chunk(sample_text, chunk_size=200, overlap=40)

print(f"Fixed-size chunking: {len(fixed_chunks)} chunks\n")
for i, chunk in enumerate(fixed_chunks):
    print(f"  Chunk {i} ({len(chunk)} chars): {chunk[:80]}...")

In [ ]:
# --- Strategy 2: Recursive Character Splitting ---

def recursive_chunk(
    text: str,
    chunk_size: int = 300,
    chunk_overlap: int = 60,
    separators: list[str] | None = None,
) -> list[str]:
    """Split text recursively on natural boundaries."""
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    chunks: list[str] = []

    def _split(text: str, seps: list[str]) -> list[str]:
        if not seps:
            return [text]
        sep = seps[0]
        if sep == "":
            parts = list(text)
        else:
            parts = text.split(sep)
        result = []
        current = ""
        for part in parts:
            candidate = current + (sep if current else "") + part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    result.append(current)
                if len(part) > chunk_size:
                    result.extend(_split(part, seps[1:]))
                    current = ""
                else:
                    current = part
        if current:
            result.append(current)
        return result

    raw = _split(text, separators)

    # Add overlap between consecutive chunks
    if chunk_overlap > 0 and len(raw) > 1:
        overlapped = [raw[0]]
        for i in range(1, len(raw)):
            prev_tail = raw[i - 1][-chunk_overlap:]
            overlapped.append(prev_tail + raw[i])
        return overlapped

    return raw


recursive_chunks = recursive_chunk(sample_text, chunk_size=300, chunk_overlap=60)

print(f"Recursive chunking: {len(recursive_chunks)} chunks\n")
for i, chunk in enumerate(recursive_chunks):
    print(f"  Chunk {i} ({len(chunk)} chars):")
    print(f"    {chunk[:100]}...")

In [ ]:
# --- Strategy 3: Semantic Chunking (embedding-based) ---
# Split where consecutive sentences have low cosine similarity

def get_embeddings(texts: list[str], model: str = "text-embedding-3-small") -> np.ndarray:
    """Embed a batch of texts using OpenAI."""
    response = client.embeddings.create(input=texts, model=model)
    return np.array([item.embedding for item in response.data])


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def semantic_chunk(
    text: str,
    similarity_threshold: float = 0.75,
) -> list[str]:
    """Split text into chunks at points where sentence similarity drops."""
    # Split into sentences
    sentences = [s.strip() for s in text.replace("\n\n", ". ").split(". ") if s.strip()]
    if len(sentences) <= 1:
        return [text]

    # Embed all sentences in one batch
    vectors = get_embeddings(sentences)

    # Find breakpoints where similarity drops below threshold
    chunks = []
    current_sentences = [sentences[0]]
    for i in range(1, len(sentences)):
        sim = cosine_similarity(vectors[i - 1], vectors[i])
        if sim < similarity_threshold:
            # Topic change detected -- start new chunk
            chunks.append(". ".join(current_sentences) + ".")
            current_sentences = [sentences[i]]
        else:
            current_sentences.append(sentences[i])
    if current_sentences:
        chunks.append(". ".join(current_sentences) + ".")

    return chunks


semantic_chunks = semantic_chunk(sample_text, similarity_threshold=0.72)

print(f"Semantic chunking: {len(semantic_chunks)} chunks\n")
for i, chunk in enumerate(semantic_chunks):
    print(f"  Chunk {i} ({len(chunk)} chars):")
    print(f"    {chunk[:120]}...\n")

In [ ]:
# --- Comparison Summary ---

print("Chunking Strategy Comparison")
print("=" * 55)
print(f"{'Strategy':<22} {'Chunks':<10} {'Avg Size':<12} {'Preserves Boundaries'}")
print("-" * 55)

for name, chunks in [
    ("Fixed-size", fixed_chunks),
    ("Recursive", recursive_chunks),
    ("Semantic", semantic_chunks),
]:
    avg = sum(len(c) for c in chunks) / len(chunks)
    preserves = "No" if name == "Fixed-size" else "Yes"
    print(f"{name:<22} {len(chunks):<10} {avg:<12.0f} {preserves}")

print("\nRecommendation: Use recursive splitting as the default.")
print("Use semantic chunking when topic coherence is critical.")

---
## Section 4: Embedding Generation with OpenAI

Embeddings are the mathematical heart of RAG. An embedding model converts text into a vector of numbers that captures **meaning**. Similar meanings produce vectors that are close together (high cosine similarity), enabling semantic search.

In [ ]:
# --- Generate Embeddings ---

texts_to_embed = [
    "How does RAG reduce hallucination?",
    "RAG retrieves relevant documents and grounds LLM outputs in source material.",
    "Fine-tuning modifies model weights using domain-specific training data.",
    "The weather in Paris is mild in spring.",
]

vectors = get_embeddings(texts_to_embed)
print(f"Embedding shape: {vectors.shape}")  # (4, 1536)
print(f"Dimension of each vector: {vectors.shape[1]}")
print(f"First 5 values of first vector: {vectors[0][:5]}")

In [ ]:
# --- Cosine Similarity Demo ---
# Show that semantically similar texts have high similarity

print("Pairwise Cosine Similarities:\n")
labels = [
    "Query: RAG + hallucination",
    "Doc: RAG retrieves docs",
    "Doc: Fine-tuning weights",
    "Doc: Weather in Paris",
]

for i in range(len(texts_to_embed)):
    for j in range(i + 1, len(texts_to_embed)):
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f"  {labels[i]}")
        print(f"    vs {labels[j]}")
        print(f"    -> similarity: {sim:.4f}")
        print()

### Embedding Model Comparison

| Model | Dimensions | MTEB Score | Cost | Best For |
|-------|-----------|------------|------|----------|
| text-embedding-3-small | 1536 | 62.3 | $0.02/1M tokens | Cost-effective production |
| text-embedding-3-large | 3072 | 64.6 | $0.13/1M tokens | Maximum quality (OpenAI) |
| BAAI/bge-large-en-v1.5 | 1024 | 64.2 | Free (local) | Privacy-first / on-prem |
| Cohere embed-v3 | 1024 | 64.5 | $0.10/1M tokens | Multilingual + compression |

> **Rule**: Always embed in batches, not one at a time. OpenAI accepts up to 2048 texts per call.

---
## Section 5: ChromaDB Vector Store Setup

ChromaDB is the simplest vector database -- an in-process store that requires no separate server. It handles embedding generation automatically when you provide an embedding function.

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# In-memory ChromaDB client (no persistence -- perfect for demos)
chroma_client = chromadb.Client()

# Create an embedding function using OpenAI
embed_fn = OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name="text-embedding-3-small",
)

# Create a collection (like a table in a relational DB)
collection = chroma_client.get_or_create_collection(
    name="rag_knowledge_base",
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"},  # Use cosine distance
)

print(f"Collection '{collection.name}' created (cosine distance)")

In [ ]:
# --- Index our sample documents ---
# ChromaDB generates embeddings automatically via the embedding function

collection.add(
    ids=[f"doc_{i}" for i in range(len(SAMPLE_DOCUMENTS))],
    documents=[doc["text"] for doc in SAMPLE_DOCUMENTS],
    metadatas=[{"source": doc["source"], "page": doc["page"]} for doc in SAMPLE_DOCUMENTS],
)

print(f"Indexed {collection.count()} documents into ChromaDB")

---
## Section 6: Similarity Search Demo

Now we can search our knowledge base by meaning, not keywords.

In [ ]:
# --- Similarity Search ---

def search_knowledge_base(query: str, n_results: int = 3) -> dict:
    """Search the ChromaDB collection and return results with metadata."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    return results


# Test queries demonstrating semantic search
test_queries = [
    "How does RAG reduce hallucination?",
    "What are the best vector databases?",
    "How should I split my documents into pieces?",
    "Should I fine-tune or use RAG?",
]

for query in test_queries:
    results = search_knowledge_base(query, n_results=2)
    print(f"Query: \"{query}\"")
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        similarity = 1 - dist  # Convert cosine distance to similarity
        print(f"  [{similarity:.3f}] {meta['source']}: {doc[:80]}...")
    print()

---
## Section 7: Basic RAG Pipeline (End-to-End)

Now we wire everything together: retrieve relevant chunks, format them into a prompt with source citations, and generate an answer with the LLM.

In [ ]:
# --- RAG System Prompt ---

RAG_SYSTEM_PROMPT = """You are an expert assistant that answers questions based ONLY on the provided context documents. Follow these rules strictly:

1. Base your answer entirely on the provided context. Do not use information from your training data.
2. If the context does not contain enough information to answer the question, say: "I don't have enough information in the available documents to answer this question."
3. Cite your sources using [Source: filename, page N] format after each claim.
4. If the context contains contradictory information, acknowledge both perspectives.
5. Keep your answer clear, well-structured, and directly responsive to the question.
6. Use the same terminology as the source documents."""


def format_context(retrieved_chunks: list[dict]) -> str:
    """Format retrieved chunks into a structured context block with source info."""
    sections = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        source = chunk.get("source", "unknown")
        page = chunk.get("page", "?")
        text = chunk["text"]
        sections.append(
            f"--- Context Document {i} [Source: {source}, Page {page}] ---\n{text}"
        )
    return "\n\n".join(sections)


def rag_query(question: str, n_context: int = 3, stream: bool = False) -> dict:
    """Complete RAG pipeline: retrieve -> format -> synthesize."""
    # Step 1: Retrieve relevant chunks
    results = search_knowledge_base(question, n_results=n_context)

    # Build chunk list with metadata
    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        chunks.append({
            "text": doc,
            "source": meta["source"],
            "page": meta["page"],
            "similarity": round(1 - dist, 4),
        })

    # Step 2: Format context
    context = format_context(chunks)

    # Step 3: Synthesize with LLM
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Context Documents:\n{context}\n\nQuestion: {question}\n\nProvide a comprehensive answer based only on the context above.",
        },
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.1,   # Low temp for factual grounding
        max_tokens=1024,
    )

    answer = response.choices[0].message.content

    return {
        "question": question,
        "answer": answer,
        "sources": [{"source": c["source"], "similarity": c["similarity"]} for c in chunks],
        "usage": {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
        },
    }


print("RAG pipeline defined. Ready to query.")

In [ ]:
# --- Run the RAG pipeline ---

result = rag_query("How does RAG reduce hallucination and how does it compare to fine-tuning?")

print("Question:", result["question"])
print("\nAnswer:")
print(textwrap.fill(result["answer"], width=88))
print("\nSources used:")
for s in result["sources"]:
    print(f"  - {s['source']} (similarity: {s['similarity']})")
print(f"\nTokens used: {result['usage']}")

In [ ]:
# --- Test with a question outside the knowledge base ---

result_unknown = rag_query("What is the capital of France?")

print("Question:", result_unknown["question"])
print("\nAnswer:")
print(textwrap.fill(result_unknown["answer"], width=88))
print("\n(The model should decline to answer since the KB has no geography info)")

---
## Section 8: Retrieval Quality Comparison

The quality of the final answer depends entirely on the quality of retrieved context. Here we compare retrieval precision across different query types.

In [ ]:
# --- Retrieval Quality Evaluation ---
# For each query, we check whether the top-K results contain the expected source.

EVAL_QUERIES = [
    {
        "query": "What is RAG and how does it work?",
        "expected_source": "rag_overview.txt",
    },
    {
        "query": "Which embedding model should I use?",
        "expected_source": "embeddings_guide.txt",
    },
    {
        "query": "How do I split documents into chunks?",
        "expected_source": "chunking_guide.txt",
    },
    {
        "query": "What vector database is best for prototyping?",
        "expected_source": "vector_db_guide.txt",
    },
    {
        "query": "How does hybrid search improve retrieval?",
        "expected_source": "retrieval_guide.txt",
    },
    {
        "query": "What temperature should I use for RAG?",
        "expected_source": "synthesis_guide.txt",
    },
]

print("Retrieval Quality Evaluation")
print("=" * 70)

hits_at_1 = 0
hits_at_3 = 0

for item in EVAL_QUERIES:
    results = search_knowledge_base(item["query"], n_results=3)
    sources = [m["source"] for m in results["metadatas"][0]]
    similarities = [round(1 - d, 3) for d in results["distances"][0]]

    hit_1 = sources[0] == item["expected_source"]
    hit_3 = item["expected_source"] in sources
    hits_at_1 += hit_1
    hits_at_3 += hit_3

    status = "HIT@1" if hit_1 else ("HIT@3" if hit_3 else "MISS")
    print(f"\n  Query: \"{item['query']}\"")
    print(f"  Expected: {item['expected_source']}")
    print(f"  Top-3 results: {list(zip(sources, similarities))}")
    print(f"  Result: {status}")

n = len(EVAL_QUERIES)
print(f"\n{'=' * 70}")
print(f"Precision@1: {hits_at_1}/{n} = {hits_at_1/n:.1%}")
print(f"Recall@3:    {hits_at_3}/{n} = {hits_at_3/n:.1%}")

---
## Section 9: Context Window Management

LLMs have a finite context window (e.g., 128K tokens for GPT-4o). We must fit the system prompt, retrieved chunks, and the question within this budget while leaving room for the answer. Overstuffing the context degrades quality.

In [ ]:
import tiktoken

# Use the tokenizer for gpt-4o / gpt-4o-mini
enc = tiktoken.encoding_for_model("gpt-4o-mini")


def count_tokens(text: str) -> int:
    """Count tokens in a string using the model's tokenizer."""
    return len(enc.encode(text))


def build_context_within_budget(
    query: str,
    n_candidates: int = 5,
    max_context_tokens: int = 3000,
    max_total_tokens: int = 4096,
) -> dict:
    """
    Retrieve chunks and fit as many as possible within the token budget.
    Returns the context string, the chunks used, and token accounting.
    """
    results = search_knowledge_base(query, n_results=n_candidates)

    system_tokens = count_tokens(RAG_SYSTEM_PROMPT)
    query_tokens = count_tokens(query)
    overhead = 80  # Formatting, delimiters, etc.
    available = max_context_tokens - system_tokens - query_tokens - overhead

    selected_chunks = []
    used_tokens = 0

    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        chunk_tokens = count_tokens(doc)
        if used_tokens + chunk_tokens > available:
            break
        selected_chunks.append({
            "text": doc,
            "source": meta["source"],
            "page": meta["page"],
            "tokens": chunk_tokens,
            "similarity": round(1 - dist, 4),
        })
        used_tokens += chunk_tokens

    return {
        "chunks_used": len(selected_chunks),
        "chunks_available": len(results["documents"][0]),
        "token_budget": {
            "system_prompt": system_tokens,
            "query": query_tokens,
            "context": used_tokens,
            "overhead": overhead,
            "total_input": system_tokens + query_tokens + used_tokens + overhead,
            "remaining_for_output": max_total_tokens - (system_tokens + query_tokens + used_tokens + overhead),
        },
        "selected_chunks": selected_chunks,
    }


# Demo: context budget management
budget_result = build_context_within_budget(
    "Explain the full RAG pipeline and all chunking strategies.",
    n_candidates=5,
    max_context_tokens=3000,
    max_total_tokens=4096,
)

print("Context Window Budget")
print("=" * 50)
for key, val in budget_result["token_budget"].items():
    print(f"  {key:<25} {val:>6} tokens")
print(f"\nChunks fit: {budget_result['chunks_used']} / {budget_result['chunks_available']}")
for c in budget_result["selected_chunks"]:
    print(f"  - {c['source']} ({c['tokens']} tokens, sim={c['similarity']})")

---
## Section 10: Citation and Source Tracking

Enterprise RAG applications require users to verify information. We implement structured citation tracking so every claim in the answer maps back to a specific source document.

In [ ]:
def rag_query_with_citations(question: str, n_context: int = 3) -> dict:
    """
    RAG pipeline that returns structured citations.
    The LLM is prompted to tag each claim with [1], [2], etc.
    """
    results = search_knowledge_base(question, n_results=n_context)

    # Build numbered source list
    sources = []
    context_parts = []
    for i, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
        start=1,
    ):
        sources.append({
            "id": i,
            "source": meta["source"],
            "page": meta["page"],
            "similarity": round(1 - dist, 4),
        })
        context_parts.append(f"[{i}] (Source: {meta['source']}, Page {meta['page']})\n{doc}")

    context_str = "\n\n".join(context_parts)

    citation_prompt = (
        "You are an expert assistant. Answer the question using ONLY the provided context. "
        "After each claim, cite the source using its number, e.g., [1]. "
        "If the context is insufficient, say so."
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": citation_prompt},
            {
                "role": "user",
                "content": f"Context:\n{context_str}\n\nQuestion: {question}",
            },
        ],
        temperature=0.1,
        max_tokens=1024,
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": sources,
    }


# Demo
cited_result = rag_query_with_citations("What are the tradeoffs between RAG and fine-tuning?")

print("Answer with Citations:")
print(textwrap.fill(cited_result["answer"], width=88))
print("\nSource Index:")
for s in cited_result["sources"]:
    print(f"  [{s['id']}] {s['source']}, page {s['page']} (similarity: {s['similarity']})")

---
## Section 11: Simple RAG with LangChain

LangChain provides high-level abstractions that wire together the entire RAG pipeline (loader, splitter, embeddings, vector store, retriever, LLM) in a few lines of code.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# 1. Create LangChain Document objects from our sample data
lc_docs = [
    Document(
        page_content=doc["text"],
        metadata={"source": doc["source"], "page": doc["page"]},
    )
    for doc in SAMPLE_DOCUMENTS
]

# 2. Chunk with RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)
lc_chunks = splitter.split_documents(lc_docs)
print(f"LangChain: {len(lc_docs)} documents -> {len(lc_chunks)} chunks")

# 3. Embed and store in Chroma (in-memory, separate from our earlier collection)
lc_vectorstore = Chroma.from_documents(
    documents=lc_chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="langchain_rag_demo",
)

# 4. Create retriever
lc_retriever = lc_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

print(f"LangChain vector store ready with {lc_vectorstore._collection.count()} chunks")

In [ ]:
# 5. Build RAG chain with a custom prompt

rag_prompt = PromptTemplate(
    template="""Use the following context to answer the question. If you cannot
answer from the context, say "I don't have that information."

Context: {context}

Question: {question}

Answer:""",
    input_variables=["context", "question"],
)

rag_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0.1),
    chain_type="stuff",  # Stuff all retrieved docs into the prompt
    retriever=lc_retriever,
    chain_type_kwargs={"prompt": rag_prompt},
    return_source_documents=True,
)

print("LangChain RetrievalQA chain built.")

In [ ]:
# 6. Query the LangChain RAG chain

lc_result = rag_chain.invoke({"query": "What is retrieval-augmented generation and why is it useful?"})

print("LangChain RAG Answer:")
print(textwrap.fill(lc_result["result"], width=88))
print("\nSource documents used:")
for doc in lc_result["source_documents"]:
    print(f"  - {doc.metadata['source']} (page {doc.metadata['page']})")
    print(f"    Preview: {doc.page_content[:80]}...")

In [ ]:
# Additional LangChain query to demonstrate versatility

lc_result2 = rag_chain.invoke({"query": "What chunking strategy should I start with?"})

print("Question: What chunking strategy should I start with?")
print("\nAnswer:")
print(textwrap.fill(lc_result2["result"], width=88))
print("\nSources:")
for doc in lc_result2["source_documents"]:
    print(f"  - {doc.metadata['source']}")

---
## Summary

In this notebook we built and explored the complete RAG pipeline:

1. **Document Loading** -- Loaded sample text documents and reviewed loader patterns for PDF, DOCX, and HTML
2. **Chunking Strategies** -- Compared fixed-size, recursive character, and semantic chunking
3. **Embedding Generation** -- Used OpenAI `text-embedding-3-small` and demonstrated cosine similarity
4. **ChromaDB Vector Store** -- Created an in-memory collection, indexed documents, and performed similarity search
5. **End-to-End RAG Pipeline** -- Built a complete retrieve-then-generate pipeline with source grounding
6. **Retrieval Quality Evaluation** -- Measured Precision@1 and Recall@3 across test queries
7. **Context Window Management** -- Token-aware budgeting to fit chunks within LLM limits
8. **Citation Tracking** -- Structured source attribution in generated answers
9. **LangChain RAG** -- Rebuilt the pipeline using LangChain's `RetrievalQA` chain

### Key Takeaways
- RAG modifies LLM generation from P(y|x) to P(y|x,z) by conditioning on retrieved evidence
- Use recursive character splitting with 512 tokens and 20% overlap as a starting point
- Always use the **same embedding model** for indexing and querying
- Use temperature 0.0-0.2 for RAG synthesis to maximize factual grounding
- Evaluate retrieval quality separately from generation quality

**Next module**: `08-advanced-rag.html` -- Advanced RAG techniques including hybrid search, reranking, query transformation, and multi-step retrieval.